## Hierachical Simulation where each loan now uses the marginal distribution implied by its FICO segment.

### 1. Imports

In [1]:
import numpy as np
import pandas as pd

from scipy.stats import lognorm

import sys

sys.path.append("../src")

from simulation import *
from dependence import *
from amortization import *




In [2]:
# Load the cleaned loan tape
SEED = 42

np.random.seed(SEED)

df = pd.read_csv(
    "../outputs/cleaned/loan_tape_clean.csv"
)

In [3]:
# create fico bands
def create_fico_band(x):

    if pd.isna(x):
        return np.nan

    if x < 580:
        return "<580"

    elif x < 670:
        return "580-669"

    elif x < 740:
        return "670-739"

    elif x < 800:
        return "740-799"

    else:
        return "800+"

df["fico_band"] = (
    df["fico_score"]
    .apply(create_fico_band)
)


In [4]:
# we draw a stratified sample of 5000 loans to speed up the simulation
sample_df = (
    df
    .groupby("fico_band", group_keys=False)
    .apply(
        lambda x: x.sample(
            frac=5000 / len(df),
            random_state=SEED
        )
    )
)

sample_df = sample_df.reset_index(drop=True)

print(sample_df.shape)

(4900, 40)


/var/folders/d4/lc69ql_15rn7y232fc6rn2nr0000gn/T/ipykernel_53365/2162664570.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [5]:
# FICO-specific Weibull parameters (final, from Part 2 §2.3)
shape_lookup = {

    "<580": 1.195284,
    "580-669": 1.386856,
    "670-739": 1.432946,
    "740-799": 1.394607,
    "800+": 1.443613
}

scale_lookup = {

    "<580": 157.805497,
    "580-669": 272.232413,
    "670-739": 260.444336,
    "740-799": 272.427402,
    "800+": 256.310957
}

# simulation settings
N_PATHS = 10000

HORIZON = 36

LGD = 0.45


U_dep = sample_one_factor_t_copula(
    
    rho=0.5602, # dependence parameter for the  t copula
    nu=2.74, # dependence parameter for the  t copula
    n_paths=N_PATHS,
    n_loans=len(sample_df),
    seed=42
)
# shape
U_dep.shape

(10000, 4900)

In [6]:
# independent benchmark
rng = np.random.default_rng(42)

U_ind = rng.uniform(
    size=U_dep.shape
)

In [7]:
# simulated default times

from scipy.stats import weibull_min

# Dependent case
T_dep = np.zeros_like(U_dep)

for j, band in enumerate(sample_df["fico_band"]):

    shape = shape_lookup[band]
    scale = scale_lookup[band]

    T_dep[:, j] = weibull_min.ppf(
        U_dep[:, j],
        c=shape,
        scale=scale
    )


# Independent case
T_ind = np.zeros_like(U_ind)

for j, band in enumerate(sample_df["fico_band"]):

    shape = shape_lookup[band]
    scale = scale_lookup[band]

    T_ind[:, j] = weibull_min.ppf(
        U_ind[:, j],
        c=shape,
        scale=scale
    )

In [8]:
# default flags
dep_default = (
    T_dep <= HORIZON
)

ind_default = (
    T_ind <= HORIZON
)

# Impute missing interest_rate with the FICO-band median before the
# amortization schedule calculation. This avoids NaN propagation in
# outstanding_balance().
sample_df["interest_rate_filled"] = (
    sample_df.groupby("fico_band")["interest_rate"]
    .transform(lambda s: s.fillna(s.median()))
)
# catch any FICO band that is entirely missing 
sample_df["interest_rate_filled"] = sample_df["interest_rate_filled"].fillna(
    sample_df["interest_rate"].median()
)


def simulated_ead_matrix(default_times, default_flags, loan_df):
    n_paths, n_loans = default_times.shape
    ead_matrix = np.zeros((n_paths, n_loans))
    issue_amt = loan_df["issue_amount"].to_numpy()
    rate = loan_df["interest_rate_filled"].to_numpy()  # <- imputed, not raw
    term = loan_df["term"].to_numpy()
    amort = loan_df["amortization_type"].to_numpy()

    for j in range(n_loans):
        mask = default_flags[:, j]
        if not mask.any():
            continue
        ead_matrix[mask, j] = [
            outstanding_balance(issue_amt[j], rate[j], term[j], amort[j], e)
            for e in default_times[mask, j]
        ]
    return ead_matrix

ead_dep = simulated_ead_matrix(T_dep, dep_default, sample_df)
ead_ind = simulated_ead_matrix(T_ind, ind_default, sample_df)

loss_dep = ead_dep * LGD
loss_ind = ead_ind * LGD

In [9]:
# ----------------------------------------
# Portfolio losses
# ----------------------------------------

portfolio_dep = loss_dep.sum(axis=1)
portfolio_ind = loss_ind.sum(axis=1)


# ----------------------------------------
# Risk measures
# ----------------------------------------

risk_dep = portfolio_risk_measures(portfolio_dep)
risk_ind = portfolio_risk_measures(portfolio_ind)

risk_table = pd.DataFrame(
    [risk_dep, risk_ind],
    index=["Student-t", "Independence"]
)

display(risk_table)


# ----------------------------------------
# MC confidence interval on VaR99.9 (dependent case)
# — previously flagged as missing; function already exists in simulation.py
# ----------------------------------------

var999_ci_dep = bootstrap_var_ci(
    portfolio_dep,
    alpha=0.999,
    n_boot=500,
    seed=SEED
)

var999_ci_ind = bootstrap_var_ci(
    portfolio_ind,
    alpha=0.999,
    n_boot=500,
    seed=SEED
)

print(f"Student-t VaR99.9 95% CI:    [{var999_ci_dep[0]:,.0f}, {var999_ci_dep[1]:,.0f}]")
print(f"Independence VaR99.9 95% CI: [{var999_ci_ind[0]:,.0f}, {var999_ci_ind[1]:,.0f}]")


# ----------------------------------------
# Diversification benefit (independence-benchmark comparison)
# ----------------------------------------

benefit = diversification_benefit(
    risk_ind["VaR999"],
    risk_dep["VaR999"]
)

print(f"\nIndependence understates EC99.9 by: {benefit:.1%}")


# ----------------------------------------
# Component VaR attribution (dependent case)
# ----------------------------------------

cvar = component_var(
    loss_dep,
    portfolio_dep,
    alpha=0.999
)

sample_df["ComponentVaR"] = cvar


# Euler additivity check — previously flagged as unverified
var999_level = np.quantile(portfolio_dep, 0.999)
euler_check = sample_df["ComponentVaR"].sum()

print(f"\nVaR99.9 (direct quantile):     {var999_level:,.0f}")
print(f"Sum of Component VaR (Euler):  {euler_check:,.0f}")
print(f"Difference:                    {euler_check - var999_level:,.0f} "
      f"({(euler_check / var999_level - 1):.2%})")


# ----------------------------------------
# Top contributors
# ----------------------------------------

top20 = (
    sample_df[["loan_id", "region", "fico_band", "issue_amount", "ComponentVaR"]]
    .sort_values("ComponentVaR", ascending=False)
    .head(20)
)

display(top20)


# ----------------------------------------
# Region attribution
# ----------------------------------------

region_risk = (
    sample_df
    .groupby("region")["ComponentVaR"]
    .sum()
    .sort_values(ascending=False)
)

display(region_risk)


# ----------------------------------------
# Concentration indices
# ----------------------------------------

exp_share = ead_dep.sum(axis=0)  # or issue_amount-based, per your Part 6 definition
exp_share = sample_df["issue_amount"].to_numpy() / sample_df["issue_amount"].sum()
HHI_exposure = np.sum(exp_share ** 2)

risk_share = sample_df["ComponentVaR"] / sample_df["ComponentVaR"].sum()
HHI_risk = np.sum(risk_share ** 2)

top10_share = (
    sample_df.nlargest(10, "ComponentVaR")["ComponentVaR"].sum()
    / sample_df["ComponentVaR"].sum()
)

diversification_ratio = (
    sample_df["ComponentVaR"].abs().sum() / var999_level
)

print(f"\nHHI (exposure):          {HHI_exposure:.4f}")
print(f"HHI (risk / ComponentVaR): {HHI_risk:.4f}")
print(f"Top-10 share of tail risk: {top10_share:.2%}")
print(f"Diversification ratio:     {diversification_ratio:.4f}")

,EL,VaR95,VaR99,VaR999,ES95,ES99,ES999,EC999
Student-t,896954.197393,4.493625e+06,8.607676e+06,1.286290e+07,7.005155e+06,1.073301e+07,1.376248e+07,1.196594e+07
Independence,895870.902760,9.790527e+05,1.013709e+06,1.048732e+06,9.998466e+05,1.031316e+06,1.069601e+06,1.528609e+05


Student-t VaR99.9 95% CI:    [12,293,164, 13,717,432]
Independence VaR99.9 95% CI: [1,043,894, 1,064,061]

Independence understates EC99.9 by: 91.8%

VaR99.9 (direct quantile):     12,862,896
Sum of Component VaR (Euler):  13,762,478
Difference:                    899,582 (6.99%)


,loan_id,region,fico_band,issue_amount,ComponentVaR
1768,ID_00009567,Northeast,740-799,12560.00,5652.00000
908,ID_00087108,Southeast,670-739,12560.00,5652.00000
2255,ID_00192476,Southwest,800+,12560.00,5652.00000
2800,ID_00293682,Pacific,<580,12560.00,5652.00000
3063,ID_00142009,Southeast,<580,12560.00,5652.00000
4273,ID_00292325,Pacific,<580,12560.00,5652.00000
3578,ID_00203153,Southwest,<580,12560.00,5652.00000
2667,ID_00122084,Southeast,<580,12560.00,5652.00000
1800,ID_00163973,Southwest,740-799,12554.21,5649.39450
4191,ID_00070153,Northeast,<580,12544.72,5645.12400


region
Southeast    3.679471e+06
Pacific      3.544238e+06
Southwest    3.516351e+06
Northeast    3.022419e+06
Name: ComponentVaR, dtype: float64


HHI (exposure):          0.0003
HHI (risk / ComponentVaR): 0.0003
Top-10 share of tail risk: 0.41%
Diversification ratio:     1.0699


In [10]:
# ============================================================
# Part 6 sensitivity: how much does EC99.9 move across the
# plausible range of nu (95% CI from Part 3: [2.05, 4.79])?
# ============================================================

nu_sensitivity_values = [2.05, 2.74, 4.79]  # CI lower bound, point estimate, CI upper bound

sensitivity_rows = []

for nu_test in nu_sensitivity_values:

    # Redraw the copula at this nu (rho held fixed at the fitted 0.5602;
    # only nu varies, isolating its effect on the tail)
    U_dep_sens = sample_one_factor_t_copula(
        rho=0.5602,
        nu=nu_test,
        n_paths=N_PATHS,
        n_loans=len(sample_df),
        seed=42
    )

    # Map through the same FICO-band Weibull marginals as the base case
    T_dep_sens = np.zeros_like(U_dep_sens)
    for j, band in enumerate(sample_df["fico_band"]):
        T_dep_sens[:, j] = weibull_min.ppf(
            U_dep_sens[:, j],
            c=shape_lookup[band],
            scale=scale_lookup[band]
        )

    dep_default_sens = T_dep_sens <= HORIZON

    # Same EAD-at-simulated-time logic as the base case
    ead_dep_sens = simulated_ead_matrix(T_dep_sens, dep_default_sens, sample_df)
    loss_dep_sens = ead_dep_sens * LGD
    portfolio_dep_sens = loss_dep_sens.sum(axis=1)

    risk_sens = portfolio_risk_measures(portfolio_dep_sens)

    sensitivity_rows.append({
        "nu": nu_test,
        "Scenario": {
            2.05: "Lower CI bound (heaviest tail)",
            2.74: "Point estimate (base case)",
            4.79: "Upper CI bound (lighter tail)"
        }[nu_test],
        "EL": risk_sens["EL"],
        "VaR99": risk_sens["VaR99"],
        "VaR999": risk_sens["VaR999"],
        "ES999": risk_sens["ES999"],
        "EC999": risk_sens["EC999"]
    })

nu_sensitivity_table = pd.DataFrame(sensitivity_rows)

display(
    nu_sensitivity_table.style.format({
        "EL": "€{:,.0f}",
        "VaR99": "€{:,.0f}",
        "VaR999": "€{:,.0f}",
        "ES999": "€{:,.0f}",
        "EC999": "€{:,.0f}"
    })
)

# Express the swing as a % of the base-case EC999, for the report narrative
base_ec999 = nu_sensitivity_table.loc[
    nu_sensitivity_table["nu"] == 2.74, "EC999"
].values[0]

nu_sensitivity_table["EC999_pct_of_base"] = (
    nu_sensitivity_table["EC999"] / base_ec999 - 1
) * 100

print(f"\nEC99.9 swing across the nu 95% CI:")
print(f"  At nu=2.05 (heaviest tail):  {nu_sensitivity_table.loc[0, 'EC999_pct_of_base']:+.1f}% vs. base case")
print(f"  At nu=2.74 (base case):   {nu_sensitivity_table.loc[1, 'EC999_pct_of_base']:+.1f}% vs. base case")
print(f"  At nu=4.79 (lightest tail):  {nu_sensitivity_table.loc[2, 'EC999_pct_of_base']:+.1f}% vs. base case")

,nu,Scenario,EL,VaR99,VaR999,ES999,EC999
0,2.050000,Lower CI bound (heaviest tail),"€904,352","€8,977,650","€13,657,520","€14,274,809","€12,753,167"
1,2.740000,Point estimate (base case),"€896,954","€8,607,676","€12,862,896","€13,762,478","€11,965,942"
2,4.790000,Upper CI bound (lighter tail),"€914,362","€8,230,040","€12,019,210","€13,254,141","€11,104,848"



EC99.9 swing across the nu 95% CI:
  At nu=2.05 (heaviest tail):  +6.6% vs. base case
  At nu=2.74 (base case):   +0.0% vs. base case
  At nu=4.79 (lightest tail):  -7.2% vs. base case
